# Dialogue Summarization with T5 on SAMSum Dataset
This notebook fine-tunes a T5 model on the SAMSum dialogue summarization dataset.
It automatically adapts to **Google Colab (CUDA GPU)**, **Mac Apple Silicon (MPS)**, or **CPU**.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

### 1. Environment & Path Setup
Automatically checks if running in Google Colab or local environment.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive'
    MODEL_SAVE_DIR = '/content/drive/MyDrive/saved_summary_model'
    print('Running on Google Colab.')
except ImportError:
    DATA_DIR = '.'
    MODEL_SAVE_DIR = './saved_summary_model'
    print('Running locally. Reading datasets from current directory.')

### 2. Load Datasets

In [ ]:
train_file = os.path.join(DATA_DIR, 'samsum-train.csv')
val_file = os.path.join(DATA_DIR, 'samsum-validation.csv')
test_file = os.path.join(DATA_DIR, 'samsum-test.csv')

train_data = pd.read_csv(train_file).dropna(subset=['dialogue', 'summary'])
val_data = pd.read_csv(val_file).dropna(subset=['dialogue', 'summary'])
test_data = pd.read_csv(test_file).dropna(subset=['dialogue', 'summary'])

print(f'Train shape: {train_data.shape}')
print(f'Val shape: {val_data.shape}')
print(f'Test shape: {test_data.shape}')

# Optional sampling for faster experimentation (set to None for full dataset)
SAMPLE_SIZE = 4000  # change to None to train on all 14,000+ samples
if SAMPLE_SIZE and SAMPLE_SIZE < len(train_data):
    train_data = train_data.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    val_data = val_data.sample(n=min(500, len(val_data)), random_state=42).reset_index(drop=True)
    print(f'Sampled Train shape: {train_data.shape}')

train_data.head()

### 3. Data Cleaning
Cleans HTML tags and excessive whitespace while preserving capitalization for proper names.

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

train_data["dialogue"] = train_data["dialogue"].apply(clean_text)
train_data["summary"] = train_data["summary"].apply(clean_text)
val_data["dialogue"] = val_data["dialogue"].apply(clean_text)
val_data["summary"] = val_data["summary"].apply(clean_text)
test_data["dialogue"] = test_data["dialogue"].apply(clean_text)
test_data["summary"] = test_data["summary"].apply(clean_text)


### 4. Tokenizer & Task Prefix Setup
*Note*: T5 is trained on multiple tasks and requires the prefix  before each dialogue.
 is great for fast local training on Mac M3, while  gives slightly higher benchmark scores.

In [ ]:
MODEL_CHECKPOINT = 't5-small'  # Use 't5-base' or 't5-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_batch(batch):
    inputs = ['summarize: ' + doc for doc in batch['dialogue']]
    targets = batch['summary']
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_pandas(train_data)
val_ds = Dataset.from_pandas(val_data)

tokenized_train = train_ds.map(preprocess_batch, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(preprocess_batch, batched=True, remove_columns=val_ds.column_names)
print('Tokenization complete. Sample sample keys:', tokenized_train[0].keys())

### 5. Load Model & Configure Device (MPS / CUDA / CPU)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

if torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Using Apple Silicon GPU via Metal Performance Shaders (MPS)')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print('Using NVIDIA CUDA GPU')
else:
    device = torch.device('cpu')
    print('Using CPU')

### 6. Training Arguments & Seq2Seq Trainer
Using  to dynamically pad sequences and automatically replace padding tokens in labels with  so cross-entropy loss ignores them.

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge_metric.compute(
        predictions=[p.strip() for p in decoded_preds],
        references=[l.strip() for l in decoded_labels],
        use_stemmer=True
    )
    return {k: round(v * 100, 4) for k, v in result.items()}

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    predict_with_generate=True,
    logging_steps=25,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


### 7. Train the Model

In [ ]:
trainer.train()

### 8. Save Fine-Tuned Model & Tokenizer

In [ ]:
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f'Model and tokenizer saved successfully to: {MODEL_SAVE_DIR}')

### 9. Test Summarization Inference

In [ ]:
# Load saved model
saved_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_SAVE_DIR).to(device)
saved_tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_DIR)

def summarize_dialogue(dialogue, max_length=128):
    saved_model.eval()
    prompt = 'summarize: ' + clean_text(dialogue)
    inputs = saved_tokenizer(prompt, return_tensors='pt', max_length=512, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = saved_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs.get('attention_mask'),
            max_new_tokens=max_length,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2
        )
    return saved_tokenizer.decode(output_ids[0], skip_special_tokens=True)

test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.
Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.
Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.
Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.
Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.
"""

summary = summarize_dialogue(test_dialogue)
print('--- GENERATED SUMMARY ---')
print(summary)